# 02 · Load reference tables
Loads `airports.csv`, `carriers.csv` and `plane_types.csv` from the data folder
(`./data` on your host, mounted read-only at `~/data`) into Iceberg tables under `local.airline`.

* Tables are **dropped and recreated** each run, so re-running never duplicates rows.
* BTS encodes missing values as `NA`, so CSVs are read with `nullValue="NA"`.
* The plane file is **`plane_types.csv`** (not `plane-data.csv`).

In [ ]:
from pyspark.sql import SparkSession

# Catalog config (REST catalog + MinIO) comes from spark-defaults.conf.
spark = SparkSession.builder.appName("02-reference-tables").getOrCreate()
spark.sparkContext.setLogLevel("WARN")
print("Spark", spark.version, "| UI:", spark.sparkContext.uiWebUrl)

def drop_table(name):
    """Drop an Iceberg table and its data files; fall back to a plain drop."""
    try:
        spark.sql(f"DROP TABLE IF EXISTS {name} PURGE")
    except Exception:
        spark.sql(f"DROP TABLE IF EXISTS {name}")

from pathlib import Path

DATA_DIR = Path.home() / "data"
NAMESPACE = "local.airline"
spark.sql(f"CREATE NAMESPACE IF NOT EXISTS {NAMESPACE}")

In [ ]:
FILES = {
    "airports":    "airports.csv",
    "carriers":    "carriers.csv",
    "plane_types": "plane_types.csv",
}

for table, fname in FILES.items():
    path = DATA_DIR / fname
    if not path.exists():
        raise FileNotFoundError(
            f"{path} not found. Put the CSVs in the host's ./data folder (see README, 'Getting the data')."
        )
    df = spark.read.csv(str(path), header=True, inferSchema=True, nullValue="NA")
    full = f"{NAMESPACE}.{table}"
    drop_table(full)
    df.writeTo(full).create()
    print(f"✅ {full:<28} {spark.table(full).count():>8,} rows")

In [ ]:
spark.table(f"{NAMESPACE}.airports").show(5, truncate=False)
spark.table(f"{NAMESPACE}.carriers").show(5, truncate=False)
spark.table(f"{NAMESPACE}.plane_types").show(5, truncate=False)